# From Pixels to Predictions
**CSI 5550 - Visual Computing**

Implement sampling, quantization, affine rotation, and histogram equalization.
Then measure how **your implementations** affect a trained image classifier.

## 1. Setup
We use ResNet-18, already trained on ImageNet, with weights supplied by
[TorchVision](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html).
No training is needed. We restrict its predictions to the ten Imagenette categories.
Images are resized/cropped to 224×224 before your operators; model normalization happens afterward.

In [ ]:
import math
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import display
from torch.utils.data import DataLoader

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "src/cnn_signal_sensitivity").exists())
sys.path.insert(0, str(ROOT / "src"))
os.environ["TORCH_HOME"] = str(ROOT / "data/models")

from cnn_signal_sensitivity import checks
from cnn_signal_sensitivity.data import CachedInputs, case_studies, load_lab_data
from cnn_signal_sensitivity.evaluation import predict, run_suite, seed_everything
from cnn_signal_sensitivity.experiments import lab_conditions
from cnn_signal_sensitivity.models import choose_device, load_classifier
from cnn_signal_sensitivity.plotting import (
    image_grid,
    plot_curves,
    show_input_output,
    transformation_gallery,
)
from cnn_signal_sensitivity.transforms import identity_condition

seed_everything(5550)
torch.set_num_threads(min(8, os.cpu_count() or 1))
DEVICE = choose_device()
OUTPUT = ROOT / "outputs/lab"
OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Using {DEVICE}")

## 2. Original images and predictions
We use the same 200 images for every comparison. Accuracy is the percentage classified
correctly; confidence is the model's probability among the ten possible categories.

In [ ]:
dataset = CachedInputs(load_lab_data(ROOT / "data"))
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=0)
model = load_classifier(DEVICE)
clean = predict(model, loader, identity_condition())
print(f"Original-image accuracy: {100 * clean.correct.mean():.1f}%")
case_images, case_labels, case_ids = case_studies(dataset)
fig = image_grid(case_images, case_labels)
plt.show()

The four operations are:

| Operation | What changes in the image |
|---|---|
| Sampling and interpolation | Reduce spatial samples, then reconstruct. |
| Intensity quantization | Use fewer intensity levels. |
| Affine rotation | Rotate and interpolate pixel values. |
| Histogram equalization | Redistribute grayscale intensities to change contrast. |

**Before running:** Which of the four operations should change predictions most? Why?

**Your prediction:** TODO

## 3. Implement the operators
Inputs are floating-point RGB tensors of shape (B,3,H,W), with values in [0,1].
Preserve shape, dtype, and device; do not change the input in place.

### A. Image sampling and interpolation
Sampling determines how finely an image's spatial detail is represented.
Keep every factor-th row and column. Reconstruct the original size using
nearest-neighbor interpolation: repeat samples, then crop to the original dimensions.
Factor 1 should reproduce the input. Use slicing and tensor repetition.

In [ ]:
def sample_image(images: torch.Tensor, factor: int) -> torch.Tensor:
    """Reduce spatial sampling, then reconstruct using nearest neighbors.

    Args:
        images: Float RGB batch in [0, 1], shaped (B, 3, H, W).
            B is the number of images; H and W are height and width.
            A single image must keep its batch dimension: (1, 3, H, W).
        factor: Positive integer spacing between retained rows and columns.

    Returns:
        Reconstructed batch with the same shape, dtype, and device as images.
        The input is not modified."""
    raise NotImplementedError("Complete sample_image before continuing.")

In [ ]:
demo_input = case_images[1:2].clone()  # one RGB image, with its batch dimension
demo_output = sample_image(demo_input, factor=8)
fig = show_input_output(demo_input, demo_output, "Output: sampling factor 8")
fig.savefig(OUTPUT / "demo_sample_image.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
checks.check_sampling(sample_image)

### B. Intensity quantization
Quantization represents pixel intensities using a finite number of levels.
Implement $Q_b(x)=\mathrm{round}[(2^b-1)x]/(2^b-1)$ for integer $1\leq b\leq8$.
Eight bits preserves the original 8-bit samples. Lower bit depths retain fewer intensity levels.

In [ ]:
def quantize(images: torch.Tensor, bits: int) -> torch.Tensor:
    """Reduce intensity resolution to 2**bits uniformly spaced levels in [0, 1].

    Args:
        images: Float RGB batch in [0, 1], shaped (B, 3, H, W).
            B is the number of images; H and W are height and width.
            A single image must keep its batch dimension: (1, 3, H, W).
        bits: Integer bit depth from 1 to 8, applied to each color channel.

    Returns:
        Quantized batch with the same shape, dtype, and device as images.
        The input is not modified."""
    raise NotImplementedError("Complete quantize before continuing.")

In [ ]:
demo_input = case_images[1:2].clone()  # one RGB image, with its batch dimension
demo_output = quantize(demo_input, bits=2)
fig = show_input_output(demo_input, demo_output, "Output: 2 bits (4 intensity levels/channel)")
fig.savefig(OUTPUT / "demo_quantize.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
checks.check_quantize(quantize)

### C. Affine geometric transformation
A geometric transformation changes pixel locations; interpolation assigns their intensities.
Rotate about the image center using inverse coordinate mapping and bilinear interpolation.
Positive angles are counterclockwise degrees. Keep the original canvas and fill outside
regions with 0.5. Construct the coordinate grid yourself; grid_sample is allowed.
See ASSIGNMENT.md for coordinate conventions.

In [ ]:
def affine_rotation(images: torch.Tensor, angle: float) -> torch.Tensor:
    """Rotate around the center using inverse mapping and bilinear interpolation.

    Args:
        images: Float RGB batch in [0, 1], shaped (B, 3, H, W).
            B is the number of images; H and W are height and width.
            A single image must keep its batch dimension: (1, 3, H, W).
        angle: Counterclockwise rotation in degrees, applied to every image.

    Returns:
        Rotated batch with the same shape, dtype, and device as images.
        Outside regions use gray (0.5). The input is not modified."""
    raise NotImplementedError("Complete affine_rotation before continuing.")

In [ ]:
demo_input = case_images[1:2].clone()  # one RGB image, with its batch dimension
demo_output = affine_rotation(demo_input, angle=30.0)
fig = show_input_output(demo_input, demo_output, "Output: 30-degree counterclockwise rotation")
fig.savefig(OUTPUT / "demo_affine_rotation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
checks.check_affine(affine_rotation)

### D. Histogram equalization
Histogram equalization redistributes intensity levels to spread contrast across their range.
Convert RGB to intensity using its channel mean. For each image, compute a 256-bin
histogram and cumulative distribution. Map intensities using (CDF−CDF_min)/(1−CDF_min).
Leave constant-intensity images unchanged. Replicate the result into three channels.

In [ ]:
def histogram_equalize(images: torch.Tensor) -> torch.Tensor:
    """Redistribute grayscale intensities using each image's cumulative histogram.

    Args:
        images: Float RGB batch in [0, 1], shaped (B, 3, H, W).
            B is the number of images; H and W are height and width.
            A single image must keep its batch dimension: (1, 3, H, W).

    Returns:
        Equalized grayscale batch in [0, 1], with intensity replicated across
        three channels. Shape, dtype, and device match images. Each image is
        equalized independently, and the input is not modified."""
    raise NotImplementedError("Complete histogram_equalize before continuing.")

In [ ]:
demo_input = case_images[1:2].clone()  # one RGB image, with its batch dimension
demo_output = histogram_equalize(demo_input)
fig = show_input_output(demo_input, demo_output, "Output: histogram-equalized grayscale")
fig.savefig(OUTPUT / "demo_histogram_equalize.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
checks.check_histogram(histogram_equalize)

## 4. Compare your results
The evaluation below calls the four functions you just wrote.
The grayscale comparison isolates histogram equalization from the removal of color.
Rotation can also crop objects; inspect the images before interpreting accuracy changes.

In [ ]:
conditions = lab_conditions(sample_image, quantize, affine_rotation, histogram_equalize)
selected = [c for c in conditions if c.key in
            {'clean', 'sampling_8', 'quantization_2', 'rotation_30', 'grayscale', 'equalized'}]
fig = transformation_gallery(model, case_images[:2], case_labels[:2], case_ids[:2], selected)
fig.savefig(OUTPUT / "examples.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
table, results = run_suite(model, loader, conditions, OUTPUT, baseline=clean,
                           metadata={"subset_manifest": "data/lab_subset.json"})
display(table[['condition', 'accuracy_pct', 'drop_pp', 'flip_pct']].round(2))
fig = plot_curves(table)
fig.savefig(OUTPUT / "accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

Accuracy drop is measured in percentage points from the original images.
Prediction flips can introduce errors or correct them.

**Discuss:** Did your results match your prediction? Explain the effects of sampling,
intensity resolution, interpolation, and contrast. Did histogram equalization help
compared with grayscale? Support your answers with numbers and examples.

**Your discussion:** TODO